In [63]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [64]:
from langchain_openai import ChatOpenAI

In [65]:
llm = ChatOpenAI(
    model="liquid/lfm-2.5-1.2b-thinking:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
)

In [66]:
response = llm.invoke("What is the meaning of life in one line?")
print(response.content)

## **RAG IMPLEMENTATION WITH OUR OWN TEXT DATA**

### **STEP 1: PREPARNG DOCUMENT FOR YOUR TEXT**

In [67]:
from langchain_core.documents import Document

my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as such: "A lot of cutting-edge AI has filtered into general applications, often without being called AI because once something becomes useful enough and common enough it's not labeled AI anymore."[2][3]

Various subfields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have adapted and integrated a wide range of techniques, including search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[4] Some companies, such as OpenAI, Google DeepMind and Meta,[5] aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human."""

docs = [Document(page_content=my_text, metadata={"source":"ABC", "id":"1"})]


### **STEP 2: Splitting the data into chunks**

In [68]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'id': '1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'id': '1'}, page_content='High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many 

In [69]:
len(chunks)

5

### **STEP 3: Creating Embeddings For The Chunks**

In [70]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4535.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [71]:
embeddings = embedding_model.embed_query("What is the meaning of life?")
print(len(embeddings))  # 384 dimensions

384


### **STEP 4: Create and Store Embeddings in Vector Store**

In [72]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### **STEP 5: Semantic Search**

In [73]:
context = vectorstore.similarity_search("What is ai?", k=3)

In [74]:
context

[Document(metadata={'id': '1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'id': '1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take acti

### **Talk to LLM**

In [76]:
response = llm.invoke(f"What is AI? You can answer using the following context: {context}")
print(response.content)

AI refers to computational systems that mimic human cognitive abilities by executing tasks like learning, reasoning, and decision-making to optimize outcomes through environmental interaction and data-driven processes, as described by the provided context.


In [77]:
print(response)

content='AI refers to computational systems that mimic human cognitive abilities by executing tasks like learning, reasoning, and decision-making to optimize outcomes through environmental interaction and data-driven processes, as described by the provided context.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 491, 'prompt_tokens': 320, 'total_tokens': 811, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 565, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'liquid/lfm-2.5-1.2b-thinking-20260120:free', 'system_fingerprint': None, 'id': 'gen-1773946896-P8OGSjC862vNbBIpdHP6',